### Libraries

In [ ]:

from pathlib import Path 
import joblib
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pandas as pd 
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay 

### Estimator Pattern: Create an estimator → fit it on training data → predict unseen examples → evaluate predictions. Never fit on the test set. 

### Validating Data

In [ ]:
DATA_PATH = Path("E:/Users/114-User23/Desktop/fakhir lab 6/data/machines_ml.csv") 
machines = pd.read_csv(DATA_PATH) 
required = {"temp_c", "vibration", "runtime", "pressure", "fault"} 
missing = required - set(machines.columns) 
if missing: 
    raise ValueError(f"Missing columns: {sorted(missing)}") 
if machines[list(required)].isna().any().any(): 
    raise ValueError("Required columns contain missing values.") 
if not set(machines["fault"].unique()).issubset({0, 1}): 
    raise ValueError("fault must contain only 0 and 1.") 

### Seperating features and columns

In [ ]:
converted_data=pd.read_csv("E:/Users/114-User23/Desktop/fakhir lab 6/data/machines_ml.csv")
df=pd.DataFrame(converted_data)
print(df.head())

### Loading dataset according to manual

In [ ]:
model = DecisionTreeClassifier(max_depth=3, random_state=42) 
X=df[['temp_c','vibration','runtime','pressure']]
Y=df['fault']
x_train,x_test,y_train,y_test= train_test_split(X,Y,test_size=0.2, stratify=Y, random_state=42)

model.fit(x_train, y_train) 
predictions = model.predict(x_test) 

### QnA
1. Why fault is excluded from X
Prevents Data Leakage: X represents the input clues, while fault is the target answer (y). Including it would give the model the answer key during training, leading to artificial 100% accuracy that collapses on real-world data.

2. Why random_state=42 makes experiments reproducible but doesn't improve accuracy
Reproducibility: It sets a fixed seed for the random number generator, ensuring data shuffling and splitting happen in the exact same way every time you run the code.
Accuracy: It only locks in the data split sequence; it does not clean data, optimize parameters, or make the model smarter.

3. Class proportions before and after the split
Maintained Balance: When using stratify=y, the percentage ratio of classes (e.g., normal vs. faulty) stays identical across your original dataset, y_train, and y_test, preventing accidental sampling bias.

4. Confirming row count compatibility
Pair Matching: X_train must match y_train in row count, and X_test must match y_test.
Total Conservation: The sum of training rows and testing rows must equal your total dataset size (len(X_train) + len(X_test) == len(df)).

### Interprating Result

In [ ]:
cm = confusion_matrix(y_test, predictions)
tn, fp, fn, tp = cm.ravel()
manual_accuracy = (tn + tp) / cm.sum()
print("Manual Accuracy") 
print({"TN": tn, "FP": fp, "FN": fn, "TP": tp}) 
test_accuracy = accuracy_score(y_test, predictions)
assert abs(manual_accuracy - test_accuracy) < 1e-12


### Plot of Confusion Matrix

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Fault'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()

### Depth Experiment and Persistence

In [ ]:
results = [] 
for depth in [1, 2, 3, 5, None]: 
    candidate = DecisionTreeClassifier(max_depth=depth, random_state=42) 
    candidate.fit(x_train, y_train) 
    train_score = candidate.score(x_train, y_train) 
    test_score = candidate.score(x_test, y_test) 
    results.append({"max_depth": str(depth), 
                    "train_accuracy%": train_score*100, 
                    "test_accuracy%": test_score*100, 
                    "gap": train_score - test_score}) 
comparison = pd.DataFrame(results) 
comparison.to_csv("E:/Users/114-User23/Desktop/fakhir lab 6/data/machines_ml.csv", index=False) 
print(comparison)

In [ ]:
results = [] 
for depth in [1, 2, 3, 5, None]: 
    candidate = DecisionTreeClassifier(max_depth=depth, random_state=42) 
    candidate.fit(x_train, y_train) 
    train_score = candidate.score(x_train, y_train) 
    test_score = candidate.score(x_test, y_test) 
    results.append({"max_depth": str(depth), 
                    "train_accuracy%": train_score*100, 
                    "test_accuracy%": test_score*100, 
                    "gap": train_score - test_score}) 
comparison = pd.DataFrame(results) 
comparison.to_csv("E:/Users/114-User23/Desktop/fakhir lab 6/data/machines_ml.csv", index=False) 
print(comparison)

### Save and Reload

In [ ]:


bundle_path = "models/machine_predictor.joblib" 

# 1. Store the column names as a clean list using .columns.tolist()
bundle = {
    "model": model, 
    "features": X.columns.tolist()
} 

# 2. Save the bundle
joblib.dump(bundle, bundle_path) 

# 3. Load the bundle back
loaded = joblib.load(bundle_path) 

# 4. Predict using the correct columns
reloaded_predictions = loaded["model"].predict(x_test[loaded["features"]]) 

# 5. Verify they match
assert (reloaded_predictions == predictions).all()
print("Success! The reloaded model predictions match perfectly.")

### Predict New Records

In [ ]:
new_machines = pd.DataFrame([ 
    {"temp_c": 98, "vibration": 6.5, "runtime": 14200, "pressure": 3.2}, 
    {"temp_c": 61, "vibration": 1.2, "runtime": 2700, "pressure": 6.7}, 
    {"temp_c": 81, "vibration": 3.8, "runtime": 8600, "pressure": 4.8}, 
]) 
feature_order = loaded["features"] 
if set(new_machines.columns) != set(feature_order): 
    raise ValueError("Prediction columns do not match training features.") 
new_machines["predicted_fault"] = loaded["model"].predict( 
    new_machines[feature_order] 
) 
print(new_machines) 

### QnA
### 1. Missing Useful Variables

* **Maintenance & Operational Factors:** Part age, time since last service, operating load capacity, and lubrication/fluid levels.
* **Environmental Factors:** Factory ambient temperature, humidity, and dust levels.

### 2. Harm Caused by Classification Errors

* **False Positive (Predicts a fault when machine is fine):**
* **Who is harmed:** The company and maintenance team.
* **Impact:** Unnecessary inspections, wasted parts, and costly production downtime.


* **False Negative (Predicts machine is fine when a fault occurs):**
* **Who is harmed:** Factory workers, machine operators, and the business.
* **Impact:** Undetected failures leading to safety hazards, catastrophic machine breakdowns, injuries, and major financial losses.